# Implement Sparse Attention on Image Tokens

**Difficulty**: 🟡 Medium

**Companies**: Meta, OpenAI, Google, DeepMind

---

### Problem Statement

Diffusion Transformers (DiT) treat an image as a grid of patch tokens and run self-attention over them. But full attention costs **O(n²)** in the number of tokens — a 256×256 image with 16×16 patches already gives **1024 tokens**, and high-resolution generation makes full attention the bottleneck.

A common trick (Swin, Neighborhood Attention, sparse DiT variants) is to exploit the **2D structure**: each token only attends to other tokens within a **local square window** on the image grid, reducing complexity to **O(n·w²)** for window size **w**.

Your task is to implement sparse 2D-window attention over image tokens by:
1. Building an attention mask from the **2D grid coordinates** of the tokens — token `(r, c)` may attend to token `(r', c')` only if both `|r - r'| <= window` and `|c - c'| <= window`.
2. Applying that mask to scaled dot-product attention over the flattened token sequence.
3. Verifying it degenerates to full attention when the window covers the whole grid, and measuring the memory savings at realistic resolutions.

---

### Requirements

1. **`create_2d_window_mask(grid_h, grid_w, window)`** — Returns a boolean mask of shape `(grid_h * grid_w, grid_h * grid_w)`. Tokens are flattened in row-major order: token index `i` corresponds to grid position `(i // grid_w, i % grid_w)`. Entry `mask[i][j]` is `True` iff token `j` lies inside the `(2*window + 1) × (2*window + 1)` neighborhood of token `i` on the grid (neighborhoods clip naturally at image boundaries).
2. **`sparse_attention(q, k, v, grid_h, grid_w, window)`** — Applies scaled dot-product attention with the 2D window mask. Masked positions must get `-inf` before softmax.
3. **Memory comparison** — Count the non-zero mask entries vs full attention for larger grids.

---

### Constraints

- ✅ Must handle batched inputs: `q, k, v` have shape `(batch_size, grid_h * grid_w, d_model)`.
- ✅ When the window covers the entire grid, output must match full (unmasked) attention exactly.
- ✅ Attention weights for each query token must sum to 1 (valid probability distribution).
- ✅ Boundary tokens (edges/corners) attend to fewer tokens — the mask must handle this without special cases in the attention itself.
- ❌ Do **not** use `torch.nn.MultiheadAttention` or any built-in attention layers.
- ❌ Do **not** use Python loops over tokens to build the mask.

---

<details>
  <summary>💡 Hint</summary>

  - Build coordinate grids with `torch.arange` + broadcasting: if `rows` is `(n, 1)` and `cols` is `(1, n)`, then `|rows_i - rows_j|` for all pairs is one broadcast subtraction away.

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F

In [ ]:
# Setup: an 8x8 grid of image patch tokens (think: 128x128 image, 16x16 patches)
torch.manual_seed(42)

batch_size = 2
grid_h, grid_w = 8, 8
num_tokens = grid_h * grid_w
d_model = 32
window = 1  # each token attends to its 3x3 neighborhood (clipped at edges)

q = torch.randn(batch_size, num_tokens, d_model)
k = torch.randn(batch_size, num_tokens, d_model)
v = torch.randn(batch_size, num_tokens, d_model)

print(f"Grid: {grid_h}x{grid_w} = {num_tokens} tokens")
print(f"Q/K/V shape: {q.shape}")
print(f"Window: {window} (interior tokens attend to {(2 * window + 1) ** 2} positions)")

In [ ]:
def create_2d_window_mask(grid_h: int, grid_w: int, window: int) -> torch.Tensor:
    """
    Create a 2D window attention mask for tokens on an image grid.

    Args:
        grid_h:  number of token rows
        grid_w:  number of token columns
        window:  neighborhood radius in each direction

    Returns:
        Boolean tensor of shape (grid_h * grid_w, grid_h * grid_w)
        where True = allowed to attend. Token index i maps to grid
        position (i // grid_w, i % grid_w).
    """
    # TODO: Compute the (row, col) coordinate of every token index
    # TODO: Build pairwise coordinate differences between all token pairs (broadcast, no loops)
    # TODO: Return True where BOTH the row and column distance are <= window
    pass


def full_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Standard scaled dot-product attention (no mask).

    Args:
        q, k, v: (batch_size, num_tokens, d_model)

    Returns:
        Output tensor of shape (batch_size, num_tokens, d_model)
    """
    # TODO: Compute scaled dot-product attention
    pass


def sparse_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                     grid_h: int, grid_w: int, window: int) -> torch.Tensor:
    """
    Scaled dot-product attention with a 2D window mask over image tokens.

    Args:
        q, k, v: (batch_size, grid_h * grid_w, d_model)
        grid_h:  number of token rows
        grid_w:  number of token columns
        window:  neighborhood radius in each direction

    Returns:
        Output tensor of shape (batch_size, grid_h * grid_w, d_model)
    """
    # TODO: Step 1 - Compute raw attention scores
    # TODO: Step 2 - Build the 2D window mask and set masked positions to -inf
    # TODO: Step 3 - Softmax and compute the weighted sum with v
    pass


# Quick sanity check: visualize the mask for one interior token
mask = create_2d_window_mask(grid_h, grid_w, window)
print(f"Mask shape: {mask.shape}")

token_idx = 3 * grid_w + 3  # token at grid position (3, 3)
print(f"Attention pattern of token (3, 3), reshaped to the grid:\n"
      f"{mask[token_idx].reshape(grid_h, grid_w).int()}")

out = sparse_attention(q, k, v, grid_h, grid_w, window)
print(f"\nOutput shape: {out.shape}")

In [ ]:
# Validation

# Validation 1: A window covering the whole grid must match full attention
print("=== Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_big_window = sparse_attention(q, k, v, grid_h, grid_w, window=max(grid_h, grid_w))
assert torch.allclose(out_full, out_big_window, atol=1e-6), "Full-grid window should match full attention!"
print("PASSED: full-grid window matches full attention.\n")

# Validation 2: Neighborhood sizes are correct (interior vs edge vs corner)
print("=== Neighborhood Sizes ===")
mask = create_2d_window_mask(grid_h, grid_w, window)
center = mask[3 * grid_w + 3].sum().item()   # token (3, 3)
edge = mask[3].sum().item()                  # token (0, 3)
corner = mask[0].sum().item()                # token (0, 0)
assert center == 9, f"Interior token should attend to 9 tokens, got {center}"
assert edge == 6, f"Edge token should attend to 6 tokens, got {edge}"
assert corner == 4, f"Corner token should attend to 4 tokens, got {corner}"
print(f"PASSED: interior=9, edge=6, corner=4 for window=1.\n")

# Validation 3: Attention weights sum to 1
print("=== Attention Weights Sum to 1 ===")
d = q.size(-1)
scores = q @ k.transpose(-2, -1) / math.sqrt(d)
scores = scores.masked_fill(~mask.unsqueeze(0), float('-inf'))
weights = F.softmax(scores, dim=-1)
weight_sums = weights.sum(dim=-1)
assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-6), "Weights must sum to 1!"
print("PASSED: All attention weight rows sum to 1.\n")

# Validation 4: Different windows give different outputs
print("=== Different Windows Give Different Outputs ===")
out_w1 = sparse_attention(q, k, v, grid_h, grid_w, window=1)
out_w2 = sparse_attention(q, k, v, grid_h, grid_w, window=2)
assert not torch.allclose(out_w1, out_w2), "Different windows should give different outputs!"
print("PASSED: window=1 and window=2 produce different outputs.\n")

# Validation 5: Memory comparison at DiT-realistic resolutions
print("=== Memory Comparison (window=3) ===")
for side in [16, 32, 64]:
    n = side * side
    m = create_2d_window_mask(side, side, 3)
    sparse_entries = m.sum().item()
    full_entries = n * n
    ratio = sparse_entries / full_entries * 100
    print(f"  grid={side}x{side} ({n:5d} tokens): {sparse_entries:>12,.0f} / {full_entries:>14,.0f} entries ({ratio:5.1f}%)")
print("PASSED: Sparse attention uses far fewer entries as resolution grows.\n")

print("All tests passed!")